# WFD low dust depth: coadded 5-sigma depth versus time, in baseline survey v5.3

This notebook follows the **coadded depth** of the **WFD low dust** region of a simulated LSST survey as the visits accumulate, band by band, and compares it with the depth of each **Deep Drilling Field (DDF)**. It is the WFD counterpart of `05_DDF_DepthVsTime`.

- creation date : 2026-10-10
- companion notebooks: `05_DDF_DepthVsTime` (depth of the DDFs), `07_WFDMiniMicroS_NVisitsVsTime` (number of visits of the WFD regions, definition of the low dust region), `02_SurveyAreas` (areas)

## Overview and outputs

1. **Low dust footprint**: HEALPix pixels (nside 64) of the scheduler target map (`CurrentAreaMap`) with the labels `lowdust` and `euclid_overlap`, as in `07_WFDMiniMicroS_NVisitsVsTime`.
2. **Load the WFD visits** from the OpSim database: all the visits except the micro-surveys (ToO, near-sun twilight, RGES, DDF), selected from the `scheduler_note` as in `07`. The visits centred outside the low dust region are kept, so that the pixels near the border of the region receive all their visits.
3. **Depth versus time of the WFD low dust**: coadded depth in each pixel, then median over the region, at 240 regular time steps.
4. **Cumulative depth of the DDFs**, same method as in `05_DDF_DepthVsTime`.
5. **Summary table**: final depth of the WFD low dust and of each DDF, and the depth gain of each DDF with respect to the WFD low dust.
6. **Figure 1**: WFD low dust alone, one curve per band.
7. **Figures 2-7**: one figure per DDF field (COSMOS, EDFS_a, EDFS_b, XMM-LSS, ELAIS-S1, ECDFS) comparing its depth (solid) with the WFD low dust depth (dashed), for the six bands.

**Method for the WFD low dust.** The WFD low dust covers a large area observed with many different pointings, so the visits cannot be coadded all together as done for a DDF (one field, one position). The coadded depth is computed **pixel by pixel**:

`m5_pix = 25 + 1.25 * log10( sum_i 10**(0.8 * (m5_i - 25)) )`

where the sum runs over the visits of one band covering the pixel (same combination as the MAF `Coaddm5Metric`). A visit covers all the pixels whose centre lies within 1.75 deg of the centre of the pixel that contains the visit centre: the dithers and the exact camera footprint are ignored. At each time step the curve is the **median** over the low dust pixels observed at least once in this band, and the shaded band is the 16th-84th percentile range of the pixels. The fraction of low dust pixels observed in each band is shown below Figure 1: at the beginning of the survey the median is computed on a small part of the region only.

**Method for the DDFs.** As in `05_DDF_DepthVsTime`: the visits of one DDF are assumed to cover the same sky position and their depths are coadded in time order. EDFS has two pointings at different sky positions (EDFS_a and EDFS_b): their visits are not coadded together, each pointing is treated as a separate field.

**Input**: the OpSim database `baseline_v5.3.6_10yrs.db` (variable `opsdb`).

**Outputs**: the tables (`.csv`) go to `DATA09_WFDLowDust_DepthVsTime/`; the figures (`.png` and `.pdf`) go to `FIG09_WFDLowDust_DepthVsTime/`.

## 0. Setup

In [ ]:
import os
import re
import sqlite3
from datetime import date, datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import healpy as hp
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.lines import Line2D

from rubin_scheduler.utils import SURVEY_START_MJD
from rubin_scheduler.scheduler.utils import CurrentAreaMap

In [ ]:
plt.rcParams.update(
    {
        "font.size": 14,
        "axes.labelsize": 16,
        "axes.titlesize": 18,
        "xtick.labelsize": 14,
        "ytick.labelsize": 14,
        "legend.fontsize": 14,
        "legend.title_fontsize": 14,
    }
)

In [ ]:
# Output directories: tables go to DATA_DIR, figures go to FIG_DIR
DATA_DIR = "DATA09_WFDLowDust_DepthVsTime"
FIG_DIR = "FIG09_WFDLowDust_DepthVsTime"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

# OpSim database. Replace by an explicit path to analyse another run.
# opsdb = get_baseline()
opsdb = "/Users/dagoret/DATA/OpSim/baseline_v5.3.6_10yrs.db"
opsim_name = os.path.split(opsdb)[-1].replace(".db", "")
print(opsdb)

# Calendar date of the survey start, used for the date axis and the year boundaries (change if needed)
SURVEY_START = date(2026, 6, 29)
N_YEARS = 10
YEAR = 365.25  # days

BANDS = list("ugrizy")

# Standard LSST band colours (the y band is drawn in black)
BAND_COLORS = {
    "u": "#4477AA",
    "g": "#228833",
    "r": "#CC3311",
    "i": "#EE7733",
    "z": "#AA3377",
    "y": "k",
}

# DDFs compared with the WFD low dust, in this order
DDF_NAMES = ["COSMOS", "EDFS_a", "EDFS_b", "XMM-LSS", "ELAIS-S1", "ECDFS"]

# WFD low dust footprint: labels of the scheduler target map (euclid_overlap is merged with lowdust, as in 07)
LOW_DUST_LABELS = ["lowdust", "euclid_overlap"]
NSIDE = 64  # HEALPix resolution of the scheduler target map and of the depth maps
FOV_RADIUS = 1.75  # deg, radius of the field of view

N_SNAP = 240  # number of time steps of the WFD depth curves (about 15 days each)
M5_REF = 25.0  # reference depth of the flux weights, avoids numerical overflow
YLIM = (22.0, 29.5)  # vertical range (mag) of the depth figures
YLIM_WFD = (22.0, 27.5)

### Utility functions

- `mjd_to_datetime`: Modified Julian Date to Python datetime.
- `survey_year_date(n)`: calendar date of the beginning of survey year n+1.
- `identify_ddf`: name of the DDF written in a `scheduler_note`, or `None` (same function as in `05_DDF_DepthVsTime`).
- `save_fig`, `add_year_markers`, `format_time_axis`: save a figure as `.png` and `.pdf`, mark the survey years, format the date axis.

In [ ]:
def mjd_to_datetime(mjd):
    # Convert a Modified Julian Date to a Python datetime
    epoch = datetime(1858, 11, 17)
    return epoch + timedelta(days=float(mjd))


def survey_year_date(n):
    # Calendar date marking the beginning of survey year n+1
    try:
        return SURVEY_START.replace(year=SURVEY_START.year + n)
    except ValueError:
        # 29 February when SURVEY_START is changed to such a date
        return SURVEY_START.replace(year=SURVEY_START.year + n, day=28)


def identify_ddf(text):
    # Identify a DDF from the scheduler_note, note or fieldName of a visit
    if not isinstance(text, str):
        return None

    s = text.upper().replace("_", "-")

    # Regular expression of each DDF name, allowing the common OpSim spellings
    aliases = {
        "COSMOS": r"COSMOS",
        "ECDFS": r"ECDFS",
        # EDFS has two pointings at different sky positions: they must not be coadded together.
        # EDFS alone is a fallback when the note does not say a or b (it then appears in the warning below).
        "EDFS_a": r"EDFS-?A(?![A-Z])",
        "EDFS_b": r"EDFS-?B(?![A-Z])",
        "EDFS": r"EDFS",
        "ELAIS-S1": r"ELAIS[- ]?S?1|ELAISS1",
        "XMM-LSS": r"XMM[- ]?LSS|XMMLSS",
    }

    # A DD tag is required, to avoid selecting an unrelated field with a similar name
    if not re.search(r"\bDD\b|DEEP.?DRILL", s):
        return None

    for name, pattern in aliases.items():
        if re.search(pattern, s):
            return name

    return None


def save_fig(fig, stem):
    # Each figure is saved as .png and .pdf in FIG_DIR
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(FIG_DIR, f"{opsim_name}_{stem}.{ext}"), dpi=180, bbox_inches="tight")


def add_year_markers(ax, label=True):
    # Dashed vertical line at the beginning of each survey year, and year label at the middle of the year
    for n in range(N_YEARS + 1):
        boundary = survey_year_date(n)
        ax.axvline(boundary, color="0.55", linestyle="--", linewidth=2, alpha=0.7)
        if label and n < N_YEARS:
            midpoint = boundary + (survey_year_date(n + 1) - boundary) / 2
            ax.text(
                midpoint,
                0.97,
                f"Y{n + 1}",
                transform=ax.get_xaxis_transform(),
                ha="center",
                va="top",
                fontsize=11,
                color="0.3",
            )


def format_time_axis(ax):
    # Date axis with one tick per year, limited to the 10 years of the survey
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.set_xlim(
        datetime.combine(SURVEY_START, datetime.min.time()),
        datetime.combine(survey_year_date(N_YEARS), datetime.min.time()),
    )


print(
    f"SURVEY_START_MJD = {SURVEY_START_MJD} -> {mjd_to_datetime(SURVEY_START_MJD).date()}; SURVEY_START set above: {SURVEY_START}"
)

## 1. WFD low dust footprint

`CurrentAreaMap` returns one survey-mode label per HEALPix pixel. The low dust pixels (`ld_pix`) are the pixels labelled `lowdust` or `euclid_overlap`.

`NEIGH` is a table giving, for each pixel, the pixels covered by a visit centred in it (all the pixels whose centre lies within `FOV_RADIUS` of the pixel centre). The lines are padded with -1. It is built once and used to spread each visit over the pixels it covers.

In [ ]:
# Scheduler footprint: one survey-mode label per HEALPix pixel
skymap = CurrentAreaMap(nside=NSIDE)
maps, labels = skymap.return_maps()
labels = np.asarray(labels).astype(str)

npix = hp.nside2npix(NSIDE)
pixarea = hp.nside2pixarea(NSIDE, degrees=True)
ld_pix = np.where(np.isin(labels, LOW_DUST_LABELS))[0]
print(f"WFD low dust: {len(ld_pix)} pixels of {pixarea:.2f} deg2 = {len(ld_pix) * pixarea:.0f} deg2")

# Pixels covered by a visit centred in each pixel: pixels with centre within FOV_RADIUS of the pixel centre
vecs = np.array(hp.pix2vec(NSIDE, np.arange(npix))).T
radius = np.radians(FOV_RADIUS)
disc_pixels = [hp.query_disc(NSIDE, vecs[p], radius) for p in range(npix)]
kmax = max(len(d) for d in disc_pixels)
NEIGH = -np.ones((npix, kmax), dtype=np.int64)
for p, d in enumerate(disc_pixels):
    NEIGH[p, : len(d)] = d
print(f"A visit covers {np.mean([len(d) for d in disc_pixels]):.1f} pixels on average (max {kmax})")

## 2. Load the WFD visits

The database is opened in read-only mode. The micro-survey visits (`ToO`, `twilight_near_sun`, `RGES`, `DD`) are removed with the same selection as in `07_WFDMiniMicroS_NVisitsVsTime`, so that the DDF visits that fall in the WFD footprint are not counted in the WFD depth. The visits are sorted in time and the HEALPix pixel of the visit centre is computed. For each band, the flux weight `10**(0.8 * (m5 - M5_REF))` of each visit is stored.

In [ ]:
def load_wfd_visits(db_path):
    # Read the non micro-survey visits of an OpSim SQLite database (read-only)
    db_path = Path(db_path).expanduser().resolve()

    if not db_path.is_file():
        raise FileNotFoundError(f"OpSim database not found: {db_path}")

    uri = db_path.as_uri() + "?mode=ro"

    with sqlite3.connect(uri, uri=True) as con:
        columns = pd.read_sql_query("PRAGMA table_info(observations)", con)["name"].tolist()

        # Band column and note column: depend on the OpSim version
        band_col = next((c for c in ("band", "filter") if c in columns), None)
        note_col = next((c for c in ("scheduler_note", "note") if c in columns), None)
        if band_col is None or note_col is None:
            raise RuntimeError(f"Missing band or note column. Columns: {columns}")

        selected = ["observationStartMJD", "fieldRA", "fieldDec", "fiveSigmaDepth", band_col, note_col]
        selected = list(dict.fromkeys(selected))
        df = pd.read_sql_query("SELECT " + ", ".join(selected) + " FROM observations", con)

    df = df.rename(columns={band_col: "filter", note_col: "note"})
    n_all = len(df)

    # Micro-surveys (same selection as 07_WFDMiniMicroS_NVisitsVsTime)
    note = df["note"].fillna("")
    is_micro = (
        note.str.contains("ToO")
        | note.str.contains("twilight_near_sun")
        | note.str.contains("RGES")
        | note.str.contains("DD")
    ).values
    df = df[~is_micro]

    df = df.dropna(subset=["observationStartMJD", "fieldRA", "fieldDec", "fiveSigmaDepth"])
    df = df[df["filter"].isin(BANDS)].sort_values("observationStartMJD").reset_index(drop=True)
    print(f"{n_all} visits in the survey, {len(df)} after removing the micro-surveys")

    # HEALPix pixel of the visit centre (RA, Dec in degrees)
    df["pix"] = hp.ang2pix(NSIDE, df["fieldRA"].values, df["fieldDec"].values, lonlat=True)
    return df


wfd = load_wfd_visits(opsdb)
print(
    f"{np.isin(labels[wfd['pix'].values], LOW_DUST_LABELS).sum()} of them are centred in the low dust region"
)

# Per band: time, pixel of the visit centre and flux weight of each visit (sorted in time)
band_data = {}
for b in BANDS:
    d = wfd[wfd["filter"] == b]
    band_data[b] = (
        d["observationStartMJD"].to_numpy(),
        d["pix"].to_numpy(),
        10.0 ** (0.8 * (d["fiveSigmaDepth"].to_numpy() - M5_REF)),
    )
    print(f"band {b}: {len(d)} visits")

## 3. Depth of the WFD low dust versus time

The survey duration is divided in `N_SNAP` equal time steps. At each step, the visits observed during the step are added to the flux sum of the pixels they cover (`flux[band]`, one value per pixel), then the statistics of the coadded depth are computed over the low dust pixels observed at least once in this band: median, 16th and 84th percentiles, and the fraction of low dust pixels observed. The result is stored in `wfd_depth` (one row per time step and band) and saved to `DATA09_WFDLowDust_DepthVsTime/` (`..._wfd_lowdust_depth_vs_time.csv`).

In [ ]:
t_last = wfd["observationStartMJD"].max()
snap_mjd = np.linspace(SURVEY_START_MJD, t_last, N_SNAP + 1)[1:]

flux = {b: np.zeros(npix) for b in BANDS}  # sum of the flux weights in each pixel
start = {b: 0 for b in BANDS}  # index of the first visit not yet added, per band
rows = []

for t_end in snap_mjd:
    for b in BANDS:
        mjd_b, pix_b, w_b = band_data[b]
        stop = np.searchsorted(mjd_b, t_end, side="right")

        # Add the visits of this time step to all the pixels they cover
        if stop > start[b]:
            idx = NEIGH[pix_b[start[b] : stop]]
            ww = np.broadcast_to(w_b[start[b] : stop, None], idx.shape)
            ok = idx >= 0
            flux[b] += np.bincount(idx[ok], weights=ww[ok], minlength=npix)
            start[b] = stop

        # Coadded depth of the low dust pixels observed at least once in this band
        f = flux[b][ld_pix]
        seen = f > 0
        if seen.any():
            m5 = M5_REF + 1.25 * np.log10(f[seen])
            med, p16, p84 = np.percentile(m5, [50, 16, 84])
        else:
            med = p16 = p84 = np.nan

        rows.append(
            {
                "mjd": t_end,
                "years": (t_end - SURVEY_START_MJD) / YEAR,
                "date": mjd_to_datetime(t_end),
                "filter": b,
                "m5_median": med,
                "m5_p16": p16,
                "m5_p84": p84,
                "n_seen": int(seen.sum()),
                "frac_seen": seen.mean(),
            }
        )

# Check: all the visits have been added
assert all(start[b] == len(band_data[b][0]) for b in BANDS), "some visits were not added"

wfd_depth = pd.DataFrame(rows)
wfd_depth.to_csv(os.path.join(DATA_DIR, f"{opsim_name}_wfd_lowdust_depth_vs_time.csv"), index=False)
wfd_depth.tail(len(BANDS))

## 4. Cumulative depth of the DDFs

Same method as in `05_DDF_DepthVsTime`: the DDF visits are identified from the `scheduler_note`, and for each (DDF, band) the single-visit depths are coadded in time order with `m5_cum = 1.25 * log10( sum_i 10**(0.8 * m5_i) )` (weights relative to the deepest visit). The result is stored in `df_cum` and saved to `DATA09_WFDLowDust_DepthVsTime/` (`..._ddf_cumulative_depth_per_visit.csv`).

In [ ]:
def load_ddf_visits(db_path):
    # Read the DDF visits of an OpSim SQLite database (read-only)
    db_path = Path(db_path).expanduser().resolve()

    if not db_path.is_file():
        raise FileNotFoundError(f"OpSim database not found: {db_path}")

    uri = db_path.as_uri() + "?mode=ro"

    with sqlite3.connect(uri, uri=True) as con:
        columns = pd.read_sql_query("PRAGMA table_info(observations)", con)["name"].tolist()

        band_col = next((c for c in ("band", "filter") if c in columns), None)
        if band_col is None:
            raise RuntimeError(f"Neither band nor filter in observations. Columns: {columns}")

        # Columns in which the name of the DDF may be written
        note_cols = [c for c in ("scheduler_note", "note", "fieldName") if c in columns]
        if not note_cols:
            raise RuntimeError("No scheduler_note, note or fieldName column.")

        selected = list(dict.fromkeys(["observationStartMJD", "fiveSigmaDepth", band_col] + note_cols))
        df = pd.read_sql_query("SELECT " + ", ".join(selected) + " FROM observations", con)

    df = df.rename(columns={band_col: "filter"})

    df["ddf"] = None
    for col in note_cols:
        df["ddf"] = df["ddf"].fillna(df[col].apply(identify_ddf))

    df["observationStartMJD"] = pd.to_numeric(df["observationStartMJD"], errors="coerce")
    df["fiveSigmaDepth"] = pd.to_numeric(df["fiveSigmaDepth"], errors="coerce")
    df = df.dropna(subset=["ddf", "observationStartMJD", "fiveSigmaDepth"])
    df = df[df["filter"].isin(BANDS)].copy()
    return df.sort_values("observationStartMJD").reset_index(drop=True)


def add_cumulative_depth(group):
    # Coadded depth after each visit, for the visits of ONE DDF in ONE band
    g = group.sort_values("observationStartMJD").copy()
    m5_max = g["fiveSigmaDepth"].max()
    weights = 10.0 ** (0.8 * (g["fiveSigmaDepth"].to_numpy() - m5_max))
    g["n_cum"] = np.arange(1, len(g) + 1)
    g["m5_cum"] = m5_max + 1.25 * np.log10(np.cumsum(weights))
    g["years"] = (g["observationStartMJD"] - SURVEY_START_MJD) / YEAR
    g["date"] = g["observationStartMJD"].apply(mjd_to_datetime)
    return g


df_ddf = load_ddf_visits(opsdb)
print(f"{len(df_ddf)} DDF visits")
print(df_ddf["ddf"].value_counts())
extra = sorted(set(df_ddf["ddf"]) - set(DDF_NAMES))
if extra:
    print("WARNING: DDFs found but not in DDF_NAMES:", extra)

df_cum = pd.concat([add_cumulative_depth(g) for _, g in df_ddf.groupby(["ddf", "filter"])])
df_cum = df_cum.sort_values("observationStartMJD").reset_index(drop=True)

cols = ["ddf", "filter", "observationStartMJD", "years", "n_cum", "fiveSigmaDepth", "m5_cum"]
df_cum[cols].to_csv(os.path.join(DATA_DIR, f"{opsim_name}_ddf_cumulative_depth_per_visit.csv"), index=False)
df_cum[cols].head()

## 5. Summary table

Final depth (mag) at the end of the survey: median coadded depth of the WFD low dust, and coadded depth of each DDF. The second table gives the **depth gain** of each DDF with respect to the WFD low dust (DDF minus WFD low dust, in mag). Both tables are saved to `DATA09_WFDLowDust_DepthVsTime/`.

In [ ]:
# Final median depth of the WFD low dust, per band
last = wfd_depth.sort_values("years").groupby("filter").tail(1).set_index("filter")
final_wfd = last["m5_median"].reindex(BANDS)

# Final coadded depth of each DDF, per band
final_ddf = df_cum.groupby(["ddf", "filter"])["m5_cum"].last().unstack("filter")
final_ddf = final_ddf.reindex(index=DDF_NAMES, columns=BANDS)

final_table = pd.concat([final_wfd.to_frame("WFD low dust (median)").T, final_ddf])
final_table.round(2).to_csv(os.path.join(DATA_DIR, f"{opsim_name}_final_depth_per_band.csv"))

gain_table = final_ddf - final_wfd
gain_table.round(2).to_csv(os.path.join(DATA_DIR, f"{opsim_name}_ddf_depth_gain_wrt_wfd_lowdust.csv"))

print("Final coadded depth m5 (mag)")
display(final_table.round(1))
print("Depth gain of each DDF with respect to the WFD low dust (mag)")
display(gain_table.round(1))

## 6. Figures

### Figure 1 - Depth of the WFD low dust versus date (`<run>_wfd_lowdust_depth.png`)

Top: median coadded depth of the low dust pixels, one curve per band, with the 16th-84th percentile range of the pixels as a shaded band. The dashed vertical lines mark the beginning of each survey year (Y1 ... Y10). Bottom: fraction of the low dust pixels observed at least once in each band; the median above is computed on these pixels only.

In [ ]:
def plot_wfd_depth(wd):
    # Top: median coadded depth of the low dust pixels. Bottom: fraction of the pixels observed.
    fig, (ax, axf) = plt.subplots(
        2,
        1,
        figsize=(12, 8),
        sharex=True,
        gridspec_kw={"height_ratios": [4, 1]},
        constrained_layout=True,
    )

    for band in BANDS:
        d = wd[wd["filter"] == band]
        ax.fill_between(d["date"], d["m5_p16"], d["m5_p84"], color=BAND_COLORS[band], alpha=0.15, linewidth=0)
        ax.plot(d["date"], d["m5_median"], color=BAND_COLORS[band], label=band, linewidth=3)
        axf.plot(d["date"], 100 * d["frac_seen"], color=BAND_COLORS[band], linewidth=2)

    add_year_markers(ax)
    add_year_markers(axf, label=False)

    ax.set_title("WFD low dust", loc="left")
    ax.set_ylabel(r"Median cumulative $m_5$ (mag)")
    ax.set_ylim(*YLIM_WFD)
    ax.grid(True, alpha=0.25)
    ax.legend(title="LSST band", ncol=6, loc="lower right", frameon=True)

    axf.set_ylabel("Pixels observed (%)")
    axf.set_xlabel("Date")
    axf.set_ylim(0, 105)
    axf.grid(True, alpha=0.25)
    format_time_axis(axf)

    fig.suptitle(f"Cumulative depth of the WFD low dust by band - {opsim_name}", fontsize=17)
    save_fig(fig, "wfd_lowdust_depth")
    return fig


fig1 = plot_wfd_depth(wfd_depth)
plt.show()

### Figures 2-7 - WFD low dust compared with each DDF (`<run>_wfd_lowdust_vs_<DDF>_depth.png`)

One figure per DDF field (COSMOS, EDFS_a, EDFS_b, XMM-LSS, ELAIS-S1, ECDFS), with the six bands. Solid curves: cumulative depth of the DDF. Dashed curves: median cumulative depth of the WFD low dust (same curves as Figure 1). The colour is the band. The DDF curves grow in steps (seasons observed in the Ocean strategy) and end far above the WFD curves, the vertical distance between a solid and a dashed curve of the same colour is the depth gain given in the summary table.

In [ ]:
def plot_compare(ddf, wd, dc):
    # Depth of one DDF (solid) and of the WFD low dust (dashed), six bands
    fig, ax = plt.subplots(figsize=(12, 5.5), constrained_layout=True)

    for band in BANDS:
        d = wd[wd["filter"] == band]
        ax.plot(d["date"], d["m5_median"], color=BAND_COLORS[band], linestyle="--", linewidth=2.5)

        v = dc[(dc["ddf"] == ddf) & (dc["filter"] == band)]
        if not v.empty:
            ax.plot(v["date"], v["m5_cum"], color=BAND_COLORS[band], linestyle="-", linewidth=3)

    add_year_markers(ax)

    # Legend: one entry per band (colour) and one per line style
    handles = [Line2D([0], [0], color=BAND_COLORS[b], lw=3, label=b) for b in BANDS]
    handles.append(Line2D([0], [0], color="0.4", lw=3, linestyle="-", label=f"{ddf} (DDF)"))
    handles.append(Line2D([0], [0], color="0.4", lw=3, linestyle="--", label="WFD low dust (median)"))
    ax.legend(handles=handles, ncol=4, loc="lower right", frameon=True)

    ax.set_title(f"{ddf} versus WFD low dust", loc="left")
    ax.set_ylabel(r"Cumulative $m_5$ (mag)")
    ax.set_xlabel("Date")
    ax.set_ylim(*YLIM)
    ax.grid(True, alpha=0.25)
    format_time_axis(ax)

    fig.suptitle(f"Cumulative depth: {ddf} and WFD low dust - {opsim_name}", fontsize=17)
    save_fig(fig, f"wfd_lowdust_vs_{ddf}_depth")
    return fig


compare_figs = {}
for ddf in DDF_NAMES:
    compare_figs[ddf] = plot_compare(ddf, wfd_depth, df_cum)
    plt.show()